# 07 — Quantum handoff builder

This stage packages augmentation methods for the shared QSVM/VQC
pipeline. It does not claim a quantum result. The canonical handoff is
6 qubits with 12 PCA features, analytic simulation, and scales n=200
and n=1000. Optional 4/8/12-qubit budgets are also exported in the
research profile for capacity analysis.

PCA is fitted once on original real training data. The shared
validation subset is identical for every method. Evaluation/test data
are not included, preventing accidental quantum-side retuning on the
final partition.

In [1]:
import hashlib, json, os, re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

PROFILE = os.getenv("REVALIDATION_PROFILE", "research").strip().lower()
REPORTABLE = PROFILE == "research"
SAMPLE_SCALES = [200] if PROFILE == "smoke" else [200, 1000]
QUBIT_BUDGETS = [6] if PROFILE == "smoke" else [4, 6, 8, 12]
CANONICAL_QUBITS = 6
FEATURES_PER_QUBIT = 2
VALIDATION_TOTAL = 200 if PROFILE == "smoke" else 1000

In [2]:
from pathlib import Path

def resolve_project_root() -> Path:
    starts = [Path.cwd().resolve()]
    try:
        starts.append(Path(__file__).resolve().parent)
    except NameError:
        pass
    for start in starts:
        for candidate in [start, *start.parents]:
            if candidate.name == "Marco_Revalidation_v2":
                return candidate
            nested = candidate / "Marco_Revalidation_v2"
            if (nested / "00_protocol").exists():
                return nested
    raise FileNotFoundError(
        "Could not locate Marco_Revalidation_v2. Run this notebook from inside "
        "the extracted project directory."
    )

PROJECT_ROOT = resolve_project_root()
CONTRACT_DIR = PROJECT_ROOT / "01_data_contract"
CLASSICAL_DIR = PROJECT_ROOT / "02_classical_augmentation" / "outputs"
CTGAN_DIR = PROJECT_ROOT / "03_ctgan"
QGAN_DIR = PROJECT_ROOT / "04_qgan"
FIDELITY_DIR = PROJECT_ROOT / "05_fidelity"
DOWNSTREAM_DIR = PROJECT_ROOT / "06_downstream"
HANDOFF_DIR = PROJECT_ROOT / "07_quantum_handoff"
REPORT_DIR = PROJECT_ROOT / "08_reporting"
print("Project root:", PROJECT_ROOT)

Project root: C:\Users\HP\Desktop\Qintern\week5\Marco_Revalidation_v2


In [3]:
TARGET = "Label"
CLASS_ORDER = ["Benign", "Ransomware", "Spyware", "Trojan"]
MINORITY_CLASSES = ["Ransomware", "Spyware", "Trojan"]
SEED = 42
EXPECTED_SHA256 = "cc7a637a174ffe797e0af0375bce3c09561f0dc8b8115c0a6292718034f5012a"

required = {
    "clean": CONTRACT_DIR / "malmem2022_clean_locked.csv.gz",
    "split": CONTRACT_DIR / "split_manifest.csv",
    "preprocessor": CONTRACT_DIR / "preprocessing.joblib",
    "arrays": CONTRACT_DIR / "locked_preprocessed_splits.npz",
    "manifest": CONTRACT_DIR / "dataset_manifest.json",
}
missing = [str(path) for path in required.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Run Notebook 01 first. Missing:\n- " + "\n- ".join(missing))

dataset_manifest = json.loads(required["manifest"].read_text(encoding="utf-8"))
preprocessing = joblib.load(required["preprocessor"])
locked = np.load(required["arrays"])
clean = pd.read_csv(required["clean"], low_memory=False)

assert dataset_manifest["source_sha256"] == EXPECTED_SHA256
assert preprocessing["dataset_sha256"] == EXPECTED_SHA256
assert len(clean) == 58_062 and clean["row_id"].is_unique
assert clean["partition"].value_counts().to_dict() == {
    "train": 40_642, "validation": 8_710, "test": 8_710
}

retained_features = list(preprocessing["retained_features"])
X_train = locked["X_train"].astype(np.float64)
X_validation = locked["X_validation"].astype(np.float64)
X_test = locked["X_test"].astype(np.float64)
y_train = locked["y_train"].astype(str)
y_validation = locked["y_validation"].astype(str)
y_test = locked["y_test"].astype(str)
assert X_train.shape == (40_642, 52)
assert X_validation.shape == (8_710, 52)
assert X_test.shape == (8_710, 52)
assert np.isfinite(X_train).all() and np.isfinite(X_validation).all() and np.isfinite(X_test).all()
print("Locked contract verified:", X_train.shape, X_validation.shape, X_test.shape)

Locked contract verified: (40642, 52) (8710, 52) (8710, 52)


## 1. Load the same selected augmentation artifacts as Notebook 06

In [ ]:
OUTPUT_DIR = HANDOFF_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
artifact_paths = {
    "SMOTE": CLASSICAL_DIR / "selected" / "selected_smote_synthetic_scaled.npz",
    "Borderline-SMOTE": CLASSICAL_DIR / "selected" / "selected_borderlinesmote_synthetic_scaled.npz",
    "ADASYN": CLASSICAL_DIR / "selected" / "selected_adasyn_synthetic_scaled.npz",
    "CTGAN": CTGAN_DIR / "outputs" / "selected" / "selected_ctgan_synthetic_scaled.npz",
    "QGAN stabilized": QGAN_DIR / "outputs" / "selected" / "selected_qgan_synthetic_scaled.npz",
}
missing = [str(p) for p in artifact_paths.values() if not p.exists()]
if missing:
    raise FileNotFoundError("Missing frozen augmentation artifacts:\n- " + "\n- ".join(missing))

datasets = {"Original": (X_train, y_train, np.zeros(len(y_train), dtype=np.uint8))}
for method, path in artifact_paths.items():
    loaded = np.load(path)
    X_syn, y_syn = loaded["X_synthetic"].astype(float), loaded["y_synthetic"].astype(str)
    datasets[method] = (
        np.vstack([X_train, X_syn]), np.concatenate([y_train, y_syn]),
        np.concatenate([np.zeros(len(y_train), np.uint8), np.ones(len(y_syn), np.uint8)]),
    )

## 2. Freeze PCA and deterministic balanced samplers

In [6]:
max_components = FEATURES_PER_QUBIT * max(QUBIT_BUDGETS)
quantum_pca = PCA(n_components=max_components, random_state=SEED).fit(X_train)
joblib.dump(quantum_pca, OUTPUT_DIR / "quantum_train_only_pca.joblib")

def balanced_indices(labels, total, seed):
    if total % len(CLASS_ORDER):
        raise ValueError("Total sample scale must be divisible by four")
    per_class = total // len(CLASS_ORDER)
    selected = []
    for class_index, class_name in enumerate(CLASS_ORDER):
        pool = np.flatnonzero(labels == class_name)
        if len(pool) < per_class:
            raise RuntimeError(f"Insufficient {class_name}: {len(pool)} < {per_class}")
        rng = np.random.default_rng(seed + 1000 * class_index)
        selected.extend(rng.choice(pool, per_class, replace=False))
    return np.asarray(selected, int)

validation_indices = balanced_indices(y_validation, VALIDATION_TOTAL, SEED + 900_000)
Z_validation_full = quantum_pca.transform(X_validation[validation_indices])
y_validation_shared = y_validation[validation_indices]
assert Counter(y_validation_shared) == Counter({c: VALIDATION_TOTAL // 4 for c in CLASS_ORDER})

## 3. Export method × scale × qubit-budget packages

In [7]:
def slug(text):
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

package_rows = []
for method_index, (method, (X_all, y_all, source_flag)) in enumerate(datasets.items()):
    Z_all = quantum_pca.transform(X_all)
    for sample_scale in SAMPLE_SCALES:
        idx = balanced_indices(y_all, sample_scale, SEED + 100_000 * method_index + sample_scale)
        for qubits in QUBIT_BUDGETS:
            feature_count = FEATURES_PER_QUBIT * qubits
            destination = OUTPUT_DIR / slug(method) / f"n_{sample_scale:04d}"
            destination.mkdir(parents=True, exist_ok=True)
            path = destination / f"q_{qubits:02d}_features_{feature_count:02d}.npz"
            np.savez_compressed(
                path,
                X_train_quantum=Z_all[idx, :feature_count].astype(np.float32),
                y_train=y_all[idx].astype(str),
                train_source_is_synthetic=source_flag[idx],
                X_validation_quantum=Z_validation_full[:, :feature_count].astype(np.float32),
                y_validation=y_validation_shared.astype(str),
                method=np.asarray([method], str),
                sample_scale=np.asarray([sample_scale], int),
                qubits=np.asarray([qubits], int),
                features_per_qubit=np.asarray([FEATURES_PER_QUBIT], int),
                seed=np.asarray([SEED], int),
            )
            digest = hashlib.sha256(path.read_bytes()).hexdigest()
            package_rows.append({
                "method": method, "sample_scale": sample_scale,
                "qubits": qubits, "feature_count": feature_count,
                "train_synthetic_rows": int(source_flag[idx].sum()),
                "validation_rows": VALIDATION_TOTAL,
                "relative_path": str(path.relative_to(PROJECT_ROOT)),
                "sha256": digest,
                "canonical": qubits == CANONICAL_QUBITS,
            })

package_table = pd.DataFrame(package_rows)
package_table.to_csv(OUTPUT_DIR / "quantum_handoff_master_table.csv", index=False)
display(package_table)

,method,sample_scale,qubits,feature_count,train_synthetic_rows,validation_rows,relative_path,sha256,canonical
0,Original,200,4,8,0,1000,07_quantum_handoff\outputs\original\n_0200\q_0...,6855202c6e8f31d0881bbff093ae373ee8bc2b14a852eb...,False
1,Original,200,6,12,0,1000,07_quantum_handoff\outputs\original\n_0200\q_0...,f7a6d395ad0923aa9a8cd78dd5e91cbad7756df98e0cd0...,True
2,Original,200,8,16,0,1000,07_quantum_handoff\outputs\original\n_0200\q_0...,a305533f9e7d711d060ac7610df04087cd008cfb548c5c...,False
3,Original,200,12,24,0,1000,07_quantum_handoff\outputs\original\n_0200\q_1...,6e17d8f8f6ffd3f816b2ffbfb64da0c07af7f062042c7d...,False
4,Original,1000,4,8,0,1000,07_quantum_handoff\outputs\original\n_1000\q_0...,76ddb0d2e7f8d321686f18d15bba2e4fbbde70da7f9c67...,False
5,Original,1000,6,12,0,1000,07_quantum_handoff\outputs\original\n_1000\q_0...,db0d356e666d21388ad9649eea5aa140fec7ea3be05bdb...,True
6,Original,1000,8,16,0,1000,07_quantum_handoff\outputs\original\n_1000\q_0...,bfc7d735187aed73f0a37dee4958a1f6ed1774e45d2638...,False
7,Original,1000,12,24,0,1000,07_quantum_handoff\outputs\original\n_1000\q_1...,09f73965e0e407b4fe17848cbd5acd833111c5b550a913...,False
8,SMOTE,200,4,8,54,1000,07_quantum_handoff\outputs\smote\n_0200\q_04_f...,299b6deaa4a0651ef5b11e68d6b83187586ffe112c387a...,False
9,SMOTE,200,6,12,54,1000,07_quantum_handoff\outputs\smote\n_0200\q_06_f...,3cdcf50add83fe9e567d4f7cc59d5b5e4675a512bb72e9...,True


## 4. Cross-file consistency checks and handoff notes

In [ ]:
for _, row in package_table.iterrows():
    loaded = np.load(PROJECT_ROOT / row.relative_path)
    assert loaded["X_train_quantum"].shape == (row.sample_scale, row.feature_count)
    assert loaded["X_validation_quantum"].shape == (VALIDATION_TOTAL, row.feature_count)
    assert Counter(loaded["y_train"].astype(str)) == Counter(
        {c: row.sample_scale // 4 for c in CLASS_ORDER}
    )
    assert np.array_equal(loaded["y_validation"].astype(str), y_validation_shared)
    assert np.isfinite(loaded["X_train_quantum"]).all()

manifest = {
    "protocol_version": "revalidation_v2_quantum_handoff_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "profile": PROFILE, "reportable": REPORTABLE,
    "dataset_sha256": EXPECTED_SHA256,
    "methods": list(datasets), "sample_scales": SAMPLE_SCALES,
    "qubit_budgets": QUBIT_BUDGETS, "canonical_qubit_budget": CANONICAL_QUBITS,
    "features_per_qubit": FEATURES_PER_QUBIT,
    "canonical_feature_dimension": CANONICAL_QUBITS * FEATURES_PER_QUBIT,
    "pca_fit": "original real training only",
    "validation_subset_shared_across_methods": True,
    "evaluation_partition_included": False,
    "backend_convention": "PennyLane default.qubit analytic shots=None",
    "seed": SEED,
    "pipeline_versions": {
        "V1": "single encoding; baseline quantum model",
        "V2": "two-pass data re-uploading with 1D ring-CNOT",
        "V3": "XGBoost feature selection / trainable kernel extensions; supplementary",
    },
    "warning": "The handoff contains inputs, not QSVM/VQC outcome claims.",
}
(OUTPUT_DIR / "quantum_handoff_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("QUANTUM HANDOFF ACCEPTED; EVALUATION PARTITION INCLUDED: False")

QUANTUM HANDOFF ACCEPTED; EVALUATION PARTITION INCLUDED: False
